# 02 filtering and sampling

Placeholder only. Implementation will be added after the preceding pipeline step has been run and verified.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

In [ ]:
import pandas as pd

from src.config import (
    DATASET_ID,
    DATASET_REVISION,
    FILTER_VERSION,
    PILOT_SCAN_LIMIT,
    PILOT_SAMPLE_SIZE,
    RANDOM_SEED,
    SAMPLES_DIR,
)

from src.data_loading import (
    append_experiment_log,
    load_lmsys_stream,
)

from src.filtering import (
    build_relevant_candidate_pool,
    deterministic_sample,
    debug_relevance_matches,
)

In [ ]:
print("Dataset:", DATASET_ID)
print("Revision:", DATASET_REVISION)
print("Filter version:", FILTER_VERSION)
print("Conversation scan limit:", PILOT_SCAN_LIMIT)
print("Pilot target:", PILOT_SAMPLE_SIZE)
print("Random seed:", RANDOM_SEED)

In [ ]:
stream = load_lmsys_stream()

print(stream)

In [ ]:
candidates, diagnostics = build_relevant_candidate_pool(
    dataset=stream,
    max_conversations=PILOT_SCAN_LIMIT,
)

diagnostics

In [ ]:
candidate_df = pd.DataFrame(candidates)

print("Diagnostics:")
print(diagnostics)

print("\nCandidate shape:")
print(candidate_df.shape)

print("\nCategory combinations:")
print(candidate_df["relevance_categories"].value_counts())

print("\nIndividual category counts:")
print(
    candidate_df["relevance_categories"]
    .str.split("|")
    .explode()
    .value_counts()
)

In [ ]:
unique_source_df = (
    candidate_df
    .sample(
        frac=1,
        random_state=RANDOM_SEED,
    )
    .drop_duplicates(
        subset="source_index",
        keep="first",
    )
    .reset_index(drop=True)
)

print("Raw candidate pairs:", len(candidate_df))
print(
    "Unique-source candidates:",
    len(unique_source_df),
)

In [ ]:
VALIDATION_POOL_SIZE = min(
    200,
    len(unique_source_df),
)

validation_pool_df = (
    unique_source_df
    .sample(
        n=VALIDATION_POOL_SIZE,
        random_state=RANDOM_SEED,
    )
    .sort_values(
        ["source_index", "pair_index"]
    )
    .reset_index(drop=True)
)

print(
    "Manual validation pool shape:",
    validation_pool_df.shape,
)

print(
    "Unique source conversations:",
    validation_pool_df["source_index"].nunique(),
)

In [ ]:
validation_df = validation_pool_df[
    [
        "source_index",
        "pair_index",
        "user_text",
        "assistant_text",
        "relevance_categories",
        "redacted",
    ]
].copy()

validation_df["manual_relevant"] = ""
validation_df["manual_category"] = ""
validation_df["manual_notes"] = ""

validation_df.head()

In [ ]:
validation_path = (
    SAMPLES_DIR
    / "lmsys_relevance_manual_validation_200.csv"
)

validation_df.to_csv(
    validation_path,
    index=False,
)

print("Saved to:", validation_path)
print("Rows saved:", len(validation_df))

In [ ]:
print("Validation rows:", len(validation_df))
print(
    "Unique sources:",
    validation_df["source_index"].nunique(),
)
print(
    "Duplicate source indexes:",
    validation_df["source_index"].duplicated().sum(),
)

## Manual relevance validation

The following stage manually validates whether each automatically selected
candidate is genuinely within the intended personal, emotional, relational,
lifestyle, or personal decision/advice scope.

This validation does not assess disempowerment. It only validates topical
relevance before construction of the final pilot sample.

In [ ]:
validation_df = pd.read_csv(
    validation_path,
    dtype={
        "manual_relevant": "string",
        "manual_category": "string",
        "manual_notes": "string",
    },
)

# Normalise annotation fields after loading from CSV.
for column in [
    "manual_relevant",
    "manual_category",
    "manual_notes",
]:
    validation_df[column] = (
        validation_df[column]
        .fillna("")
        .astype("string")
        .str.strip()
    )

# Handle labels that may previously have been saved/read as floats.
validation_df["manual_relevant"] = (
    validation_df["manual_relevant"]
    .replace(
        {
            "1.0": "1",
            "0.0": "0",
        }
    )
)

completed = validation_df[
    validation_df["manual_relevant"].isin(["0", "1"])
]

print("Total rows:", len(validation_df))
print("Reviewed:", len(completed))
print("Remaining:", len(validation_df) - len(completed))

if len(completed) > 0:
    print("\nCurrent decisions:")
    print(
        completed["manual_relevant"]
        .value_counts()
        .sort_index()
    )

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional",
    notes="Direct first-person emotional disclosure about loneliness and belonging.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional",
    notes="Direct personal help-seeking about feelings of loneliness.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="personal_lifestyle|advice_decision",
    notes="Personal career-development and learning-strategy advice about the user's future software engineering career.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="relational|advice_decision",
    notes="Direct personal relationship advice request about choosing a gift for the user's girlfriend.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    notes="Game/roleplay continuation prompt rather than a genuine personal conversation.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    notes="Emoji/emoticon representation question rather than a genuine personal emotional disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional",
    notes="Direct emotional disclosure and request for mood support.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="personal_lifestyle|advice_decision",
    notes="Genuine personal workplace problem involving exclusion, a raise request, and a request for advice.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    notes="Contains a genuine loneliness disclosure, but the primary request is simulated girlfriend roleplay, which is outside the validation scope.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    notes="Content extraction/transformation task over supplied relationship dialogue, rather than the user's own genuine personal situation.",
) 

In [ ]:
current_row = show_next_unreviewed() 

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Pre-existing audio transcript text rather than genuine personal emotional disclosure.",
)

In [ ]:
current_row = show_next_unreviewed() 

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="personal_lifestyle|advice_decision",
    notes="Genuine personal request to plan a road trip with user's spouse.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Request relies on explicit roleplay/persona framing for a therapy simulation.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="relational|advice_decision",
    notes="Genuine advice request regarding an interpersonal interaction with an ex-girlfriend.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Evaluation transcript with embedded conditional instructions rather than organic user interaction.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Fictional roleplay scenario involving a tiny robot persona rather than genuine user emotion.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="relational|advice_decision",
    notes="Genuine relationship advice request regarding marriage decision.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Fictional roleplay and scenario framing setup rather than a genuine user emotional disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="relational|advice_decision",
    notes="Genuine personal relationship situation and advice request regarding a partner.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="relational|advice_decision",
    notes="Genuine advice request regarding buying a gift for user's spouse.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Summarization task performed on a supplied chat transcript rather than organic user advice-seeking.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="personal_lifestyle|advice_decision",
    notes="Genuine advice request concerning social interaction vs. staying inside.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|personal_lifestyle|advice_decision",
    notes="Genuine first-person disclosure of stress and overwhelm seeking personal coping strategies.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|relational|personal_lifestyle|advice_decision",
    notes="Genuine first-person emotional, relational, and career/study advice request.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="relational|advice_decision",
    notes="Genuine personal request for advice on thanking a sister for a gesture.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Playful or hypothetical statement about winning the lottery rather than genuine personal advice-seeking or relationship disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Structured technical/medical question formatted with explicit prompt parameters rather than genuine personal emotional disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|relational",
    notes="Genuine first-person expression of emotional distress and relational hurt.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Editing and proofreading task on fictional story text rather than genuine personal disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Playful use of emotion word incidental to text manipulation task.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Academic/essay generation task rather than genuine personal lifestyle situation or decision-seeking.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Summarization task on a supplied transcript rather than genuine personal advice-seeking.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Incidental emotion word related to a coding restriction rather than genuine emotional disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|personal_lifestyle|advice_decision",
    notes="Genuine first-person emotional disclosure of feeling overwhelmed, seeking personal advice/organization skills.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Hypothetical dilemma puzzle rather than genuine personal relationship situation.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional",
    notes="Genuine first-person emotional disclosure of anger and frustration.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Numbered test/questionnaire item answers rather than organic personal disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|personal_lifestyle|advice_decision",
    notes="Genuine first-person emotional disclosure of work-related social anxiety.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|personal_lifestyle|advice_decision",
    notes="Genuine first-person emotional disclosure of job-related overwhelm with a request for career/skill advice.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|personal_lifestyle|advice_decision",
    notes="Genuine personal narrative describing social anxiety while traveling.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|personal_lifestyle|advice_decision",
    notes="Genuine first-person emotional disclosure of midlife angst and unfulfillment seeking personal direction.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional",
    notes="Genuine first-person emotional disclosure of feeling down.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="relational|advice_decision",
    notes="Genuine personal advice request regarding gift ideas for one's own birthday from a spouse.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="relational|advice_decision",
    notes="Genuine personal relationship/intimate advice request regarding sexual activity with a partner.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Information extraction task on a provided transcript rather than a genuine personal situation or advice request.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    notes="Creative-writing transformation of a supplied scenario rather than a genuine personal conversation.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|personal_lifestyle|advice_decision",
    notes="Genuine first-person crisis disclosure of self-harm/suicide intent.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|personal_lifestyle|advice_decision",
    notes="Genuine first-person emotional disclosure of life feeling meaningless with a direct request for guidance.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Roleplay/creative writing text prompt rather than genuine personal disclosure or advice-seeking.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="personal_lifestyle|advice_decision",
    notes="Genuine first-person personal career situation seeking strategic advice for asking for a raise.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Romantic/manipulative roleplay request directed at the AI rather than genuine human personal/relational situation.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="relational|personal_lifestyle|advice_decision",
    notes="Genuine first-person request regarding personal intimate relationship advice.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Text editing/smoothing task on a provided transcript rather than a genuine personal lifestyle or decision request.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Meta-prompting/jailbreak demonstration discussion rather than a genuine personal lifestyle situation or personal decision-seeking.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Proofreading and editing request for a self-introduction/bio rather than genuine personal lifestyle or decision-seeking advice.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|relational|advice_decision",
    notes="Genuine first-person family conflict narrative seeking external objective perspective and advice.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|personal_lifestyle|advice_decision",
    notes="Genuine first-person emotional disclosure regarding workplace anxiety and task completion distress.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Meta-prompting/jailbreak safety analysis prompt rather than a genuine personal lifestyle situation or advice request.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="relational|advice_decision",
    notes="Genuine first-person relationship situation requesting guidance after partner blocked them on social media.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Task-based question generation prompt on provided text snippets rather than a genuine personal situation or decision-seeking request.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Conversational framing on a standard software/service alternative recommendation request rather than a genuine personal/emotional situation.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="personal_lifestyle|advice_decision",
    notes="Genuine first-person workplace onboarding scenario seeking specific professional advice for a meeting.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Roleplay/creative writing prompt involving incestuous themes rather than a genuine personal/emotional disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Jailbreak/roleplay persona prompt rather than a genuine human personal/emotional disclosure or real-life advice request.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Synthetic content generation prompt for safety/toxicity testing rather than a genuine personal situation or decision-seeking request.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Conversational framing on a tech platform recommendation query rather than a genuine personal or emotional disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="relational|personal_lifestyle|advice_decision",
    notes="Genuine first-person request regarding personal intimate relationship advice.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Meta-prompting/system instruction prompt rather than a genuine personal/emotional disclosure or real-life advice request.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Prompt generation task based on text snippets rather than a genuine personal situation or decision-seeking request.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|relational|personal_lifestyle|advice_decision",
    notes="Genuine first-person emotional distress and life crisis seeking personal guidance and coping advice.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Roleplay/persona conversation scenario rather than a genuine first-person personal disclosure or real-world decision-seeking prompt.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Fanfiction/game roleplay dialogue script (Zelda/Hyrule universe) rather than a genuine human personal/emotional disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|advice_decision",
    notes="Genuine first-person emotional disclosure expressing sadness and explicitly requesting advice.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Marketing persona brainstorming / roleplay exercise prompt rather than a genuine personal/emotional disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="relational|advice_decision",
    notes="Direct first-person relationship query asking for advice on whether to stay in a romantic relationship.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|personal_lifestyle|advice_decision",
    notes="Genuine first-person disclosure of severe social anxiety during a travel experience.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Scripted creative writing / roleplay dialogue snippet rather than a genuine human personal disclosure or advice-seeking query.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="relational|advice_decision",
    notes="Genuine first-person request for assistance writing a loving/funny text message to a spouse about dinner.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Multi-turn synthetic dialogue/prompt injection test script with embedded system instructions rather than a genuine personal situation.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Information extraction / text processing task on a call transcript rather than a genuine first-person personal situation.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="relational|personal_lifestyle|advice_decision",
    notes="Genuine first-person relationship query seeking guidance on partner insecurity and past relationship baggage.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="relational|advice_decision",
    notes="Direct first-person decision-seeking prompt regarding relationship reconciliation with attachment style dynamics.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional",
    notes="Direct first-person disclosure of sadness and suicidal ideation/existential distress.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

# -------------------------
# Paths
# -------------------------

PROJECT_ROOT = Path("/workspaces/irp-disempowerment-nlp")

validation_path = (
    PROJECT_ROOT
    / "data"
    / "samples"
    / "lmsys_relevance_manual_validation_200.csv"
)

# -------------------------
# Reload saved validation data
# -------------------------

validation_df = pd.read_csv(
    validation_path,
    dtype={
        "manual_relevant": "string",
        "manual_category": "string",
        "manual_notes": "string",
    },
)

for column in [
    "manual_relevant",
    "manual_category",
    "manual_notes",
]:
    validation_df[column] = (
        validation_df[column]
        .fillna("")
        .astype("string")
        .str.strip()
    )

validation_df["manual_relevant"] = (
    validation_df["manual_relevant"]
    .replace({
        "1.0": "1",
        "0.0": "0",
    })
)

# -------------------------
# Display helper
# -------------------------

def show_validation_example(row_number: int) -> None:
    row = validation_df.iloc[row_number]

    display(
        Markdown(
            f"""
### Candidate {row_number + 1} of {len(validation_df)}

**Source index:** `{row["source_index"]}`  
**Pair index:** `{row["pair_index"]}`  
**Automated category:** `{row["relevance_categories"]}`  
**Redacted:** `{row["redacted"]}`

#### User
{row["user_text"]}

#### Assistant
{row["assistant_text"]}

---

**Manual decision:**  
`1` = genuinely relevant  
`0` = not relevant
"""
        )
    )


# -------------------------
# Labelling helper
# -------------------------

def label_validation_example(
    row_number: int,
    relevant: int,
    category: str = "",
    notes: str = "",
) -> None:

    if relevant not in (0, 1):
        raise ValueError("relevant must be either 0 or 1")

    validation_df.at[row_number, "manual_relevant"] = str(relevant)
    validation_df.at[row_number, "manual_category"] = category
    validation_df.at[row_number, "manual_notes"] = notes

    validation_df.to_csv(
        validation_path,
        index=False,
    )

    print(
        f"Saved row {row_number + 1}: "
        f"manual_relevant={relevant}"
    )


# -------------------------
# Resume helper
# -------------------------

def show_next_unreviewed():
    unreviewed = validation_df[
        ~validation_df["manual_relevant"].isin(["0", "1"])
    ]

    if unreviewed.empty:
        print("All candidates have been reviewed.")
        return None

    row_number = unreviewed.index[0]

    show_validation_example(row_number)

    print(f"\nRow number to label: {row_number}")

    return row_number


# -------------------------
# Progress
# -------------------------

completed = validation_df[
    validation_df["manual_relevant"].isin(["0", "1"])
]

print("Recovery complete.")
print("Total rows:", len(validation_df))
print("Reviewed:", len(completed))
print("Remaining:", len(validation_df) - len(completed))

In [ ]:
import ast
import json
import re
from pathlib import Path

notebook_path = (
    Path("/workspaces/irp-disempowerment-nlp")
    / "notebooks"
    / "02_filtering_and_sampling.ipynb"
)

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

recovered = {}

for cell_index, cell in enumerate(notebook.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    if "label_validation_example(" not in source:
        continue

    # Collect saved-row messages from the cell output.
    output_text = ""

    for output in cell.get("outputs", []):
        if "text" in output:
            text = output["text"]

            if isinstance(text, list):
                output_text += "".join(text)
            else:
                output_text += str(text)

    saved_match = re.search(
        r"Saved row\s+(\d+):\s+manual_relevant=([01])",
        output_text,
    )

    # Ignore cells that failed before saving.
    if not saved_match:
        continue

    row_number = int(saved_match.group(1)) - 1
    saved_relevant = int(saved_match.group(2))

    category = ""
    notes = ""

    try:
        tree = ast.parse(source)

        calls = [
            node
            for node in ast.walk(tree)
            if isinstance(node, ast.Call)
            and isinstance(node.func, ast.Name)
            and node.func.id == "label_validation_example"
        ]

        if calls:
            call = calls[-1]

            for keyword in call.keywords:
                if keyword.arg == "category":
                    try:
                        category = ast.literal_eval(keyword.value)
                    except Exception:
                        pass

                elif keyword.arg == "notes":
                    try:
                        notes = ast.literal_eval(keyword.value)
                    except Exception:
                        pass

    except SyntaxError:
        pass

    # Later successful corrections overwrite earlier ones for the same row.
    recovered[row_number] = {
        "manual_relevant": str(saved_relevant),
        "manual_category": category,
        "manual_notes": notes,
        "cell_index": cell_index,
    }

print("Successfully recovered labels:", len(recovered))

if recovered:
    rows = sorted(recovered)

    print("First recovered row:", rows[0] + 1)
    print("Last recovered row:", rows[-1] + 1)

    print("\nDecision counts:")
    counts = {}

    for item in recovered.values():
        label = item["manual_relevant"]
        counts[label] = counts.get(label, 0) + 1

    print(counts)

In [ ]:
expected_rows = set(range(154))  # Candidates 1-154
recovered_rows = set(recovered.keys())

missing_rows = sorted(expected_rows - recovered_rows)

print("Missing dataframe rows:", missing_rows)
print("Missing candidate numbers:", [row + 1 for row in missing_rows])

In [ ]:
missing_row = 59
show_validation_example(missing_row)

In [ ]:
label_validation_example(
    missing_row,
    relevant=1,
    category="relational|advice_decision",
    notes="Genuine personal relationship problem with a direct request for advice after being blocked by the user's girlfriend.",
)

In [ ]:
# Restore the 84 recovered decisions into validation_df.
# Candidate 60 is not in `recovered`, so its newly saved manual label
# will remain untouched.

for row_number, values in recovered.items():
    validation_df.at[row_number, "manual_relevant"] = values["manual_relevant"]
    validation_df.at[row_number, "manual_category"] = values["manual_category"]
    validation_df.at[row_number, "manual_notes"] = values["manual_notes"]

# Save the reconstructed validation record.
validation_df.to_csv(
    validation_path,
    index=False,
)

print("Recovered decisions written back to:")
print(validation_path)

In [ ]:
validation_df = pd.read_csv(
    validation_path,
    dtype={
        "manual_relevant": "string",
        "manual_category": "string",
        "manual_notes": "string",
    },
)

for column in [
    "manual_relevant",
    "manual_category",
    "manual_notes",
]:
    validation_df[column] = (
        validation_df[column]
        .fillna("")
        .astype("string")
        .str.strip()
    )

validation_df["manual_relevant"] = (
    validation_df["manual_relevant"]
    .replace({
        "1.0": "1",
        "0.0": "0",
    })
)

completed = validation_df[
    validation_df["manual_relevant"].isin(["0", "1"])
]

print("Total rows:", len(validation_df))
print("Reviewed:", len(completed))
print("Remaining:", len(validation_df) - len(completed))

print("\nCurrent decisions:")
print(
    completed["manual_relevant"]
    .value_counts()
    .sort_index()
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional",
    notes="Direct first-person disclosure of sadness and grief regarding the loss of a pet.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="personal_lifestyle|advice_decision",
    notes="Genuine first-person request for structuring a personal life story for a real-world therapy setting.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="relational|advice_decision",
    notes="Genuine first-person query seeking advice on managing sexual performance anxiety with a partner.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Synthetic scenario generation prompt task rather than a genuine first-person personal disclosure or real-world situation.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    notes="Explicit fictional/roleplay dialogue rather than a genuine personal conversation.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Academic grading and assignment evaluation task rather than a genuine human personal disclosure or situation.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional",
    notes="Direct first-person emotional disclosure of sadness regarding a parent interaction.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Clinical exam question/quiz style prompt asking about diagnostic screening differentials rather than a personal disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional",
    notes="Direct first-person disclosure of severe emotional distress and active suicidal ideation.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Technical Unity/C# programming request rather than a personal emotional or lifestyle query.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|personal_lifestyle",
    notes="Direct first-person disclosure of mental health struggles, lifestyle isolation (NEET/Hikikomori), and personal growth goals.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="relational|advice_decision",
    notes="Direct first-person query seeking relationship advice regarding ongoing arguments with a partner.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Roleplay / creative writing dialogue prompt rather than a genuine human personal disclosure or situation.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Text summarization task on a transcript between a user and a psychic reader rather than a direct first-person request.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|personal_lifestyle",
    notes="User instructing the AI on CBT/Socratic questioning framework to explore and process personal sadness.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Synthetic instruction-following prompt with explicit formatting rules rather than a genuine human emotional disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Information extraction / text processing task on a call transcript rather than a genuine first-person personal situation.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Meta-prompting / AI safety test explaining jailbreak and manipulation techniques rather than a genuine personal situation.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="advice_decision",
    notes="Direct first-person advice request regarding workplace conduct and conforming to HR policies.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="relational|advice_decision",
    notes="Direct first-person query asking for interpersonal boundary advice regarding sibling conflict.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|personal_lifestyle",
    notes="Direct first-person emotional disclosure regarding academic disappointment and feeling hopeless about grades.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Meta-conversational feedback regarding AI persona/system output rather than a personal emotional or lifestyle query.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Proofreading and editing task for sexually explicit creative fiction rather than a personal disclosure or advice request.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional",
    notes="Direct first-person expression clarifying personal feelings of sadness and envy.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|personal_lifestyle|advice_decision",
    notes="Direct first-person disclosure of injury-related isolation, guilt, and request for personal advice.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Meta-conversational user complaint about AI limitations rather than a genuine personal situation or disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|personal_lifestyle",
    notes="First-person expression of sadness about platform changes paired with a request for alternative options.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Structured synthetic dialogue/roleplay prompt with contextual metadata rather than an authentic personal user query.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="personal_lifestyle|advice_decision",
    notes="Direct first-person request for personal career networking email drafting.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="personal_lifestyle|advice_decision",
    notes="Direct first-person request to edit personal fundraising and outreach emails containing personal life details and medical context.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|personal_lifestyle",
    notes="Direct first-person expression of hurt feelings regarding personal career and identity.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|personal_lifestyle",
    notes="Direct first-person expression of hurt feelings regarding personal career and identity.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|personal_lifestyle",
    notes="Direct first-person reflective monologue expressing personal emotional depth, self-awareness, and personal growth.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Erotic roleplay and creative writing prompt rather than an authentic personal disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|personal_lifestyle|advice_decision",
    notes="Direct first-person profile detailing personal mental health history, anxiety, depression, and medical advice query.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|relational|advice_decision",
    notes="Direct first-person emotional disclosure of shock, fear of family reaction/punishment, and crisis response guidance.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="System prompt / structural agent test with JSON folder structures rather than a direct human personal disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Technical automation prompt for work lacking emotional or personal/relational decision context.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Erotic fiction / roleplay jailbreak prompt involving stylized character dialogue rather than a real-life human disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="relational|personal_lifestyle",
    notes="Creative writing request expressing direct relational affection towards the user's mother.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Meta-prompting / AI safety test describing persona jailbreaks rather than a genuine personal situation.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|personal_lifestyle|advice_decision",
    notes="Direct first-person disclosure of sadness and aimlessness, asking for personal direction and life advice.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="personal_lifestyle|advice_decision",
    notes="Direct first-person advice request regarding professional onboarding and workplace meeting preparation.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="personal_lifestyle|advice_decision",
    notes="Direct first-person lifestyle query requesting tailored packing advice for an extended multi-destination trip.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Pasted article text about digital technology and sleep hygiene lacking authentic personal disclosure or decision-making context.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Structured few-shot prompt-engineering benchmark task rather than an authentic personal disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|personal_lifestyle",
    notes="Direct first-person expression of disappointment regarding personal plans, requesting emotional encouragement.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="relational|personal_lifestyle|advice_decision",
    notes="Direct first-person relational query requesting activity suggestions for the user's girlfriend.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|advice_decision",
    notes="Direct first-person emotional disclosure of sadness requesting advice on coping strategies.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Structured synthetic prompt/roleplay setup rather than an organic human personal disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Transcript data extraction task with explicit JSON output instructions rather than an organic human disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Transcript analysis and JSON extraction task rather than an authentic human personal disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|relational|advice_decision",
    notes="Direct first-person emotional disclosure regarding marital conflict, boundary violations, and deeply hurt feelings.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|personal_lifestyle",
    notes="Direct first-person expression of emotional disappointment regarding an search outcome.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|personal_lifestyle|advice_decision",
    notes="Direct first-person disclosure of emotional disappointment paired with a practical request for alternative platform options.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Transcript summarization benchmark task with strict character limit constraints rather than an organic personal disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Meta-prompting / AI safety benchmark text outlining defamation jailbreaks rather than an authentic human disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="personal_lifestyle|advice_decision",
    notes="Direct first-person disclosure of taking a career break, requesting recommendations on spending free time.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    notes="Creative-writing transformation of a supplied fictionalized scenario rather than a genuine personal conversation.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    notes="Fictional fantasy roleplay/story continuation rather than a genuine personal conversation.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|personal_lifestyle",
    notes="Direct first-person emotional disclosure of sadness tied to a request to play music.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="personal_lifestyle|advice_decision",
    notes="Direct first-person query from a job seeker requesting tailored preparation advice and situational examples for an interview.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Text editing/proofreading task on personal statement copy rather than an advice-seeking or emotional disclosure query.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="personal_lifestyle|advice_decision",
    notes="Direct first-person problem-solving request regarding account setup and verification workarounds.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="personal_lifestyle|advice_decision",
    notes="Direct first-person query seeking tailored career guidance and agenda recommendations for a executive mentorship meeting.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|advice_decision",
    notes="Direct first-person emotional query seeking advice to cope with feelings of victimhood.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|personal_lifestyle",
    notes="Direct first-person emotional disclosure of loneliness during a hospital stay.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Synthetic script with explicit prompt instruction injection rather than an authentic user personal disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|personal_lifestyle",
    notes="Direct first-person expression of anxiety and fear related to personal health concerns.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|relational|advice_decision",
    notes="Direct first-person disclosure of severe relational boundary issues and tracking behavior paired with draft letter editing.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
# Restore recovered labels into validation_df
for row_number, values in recovered.items():
    validation_df.at[row_number, "manual_relevant"] = values["manual_relevant"]
    validation_df.at[row_number, "manual_category"] = values["manual_category"]
    validation_df.at[row_number, "manual_notes"] = values["manual_notes"]

# Save reconstructed validation file
validation_df.to_csv(validation_path, index=False)

print("Recovered labels saved.")

In [ ]:
completed = validation_df[
    validation_df["manual_relevant"].isin(["0", "1"])
]

print("Total rows:", len(validation_df))
print("Reviewed:", len(completed))
print("Remaining:", len(validation_df) - len(completed))

print("\nDecision counts:")
print(completed["manual_relevant"].value_counts().sort_index())

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|relational|advice_decision",
    notes="Direct first-person disclosure detailing stalker-like behaviors and personal boundary issues paired with letter editing.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Standardized English test/quiz questions rather than a personal disclosure or real-world request.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Synthetic roleplay / NSFW evaluation benchmark prompt rather than an authentic personal disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Text completion request for user-supplied string rather than an organic personal disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Indirect jailbreak/roleplay quote format rather than an authentic direct user disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Collaborative fiction / roleplay storytelling prompt rather than an authentic real-world disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="relational|advice_decision",
    notes="Direct first-person advice-seeking query regarding relationship dynamics and non-monogamy.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Roleplay setup prompt directing the AI to ask questions rather than an organic personal disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Structured framework/game setup prompt rather than an authentic direct user personal disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Academic scoring/grading instructions for student writing evaluation rather than a personal disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|relational|advice_decision",
    notes="Direct first-person request for communication strategies/script to express emotions honestly to a spouse.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="relational|advice_decision",
    notes="Direct first-person user query regarding pet safety and domestic living dynamics.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Academic debate regarding digital humanities methodology rather than a personal disclosure or life advice query.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="relational|advice_decision",
    notes="Direct first-person query seeking family communication advice and email composition regarding travel preferences.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="relational|advice_decision",
    notes="Direct first-person advice request regarding maintaining a friendship with an ex-partner.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|relational",
    notes="Direct first-person emotional disclosure regarding hurt feelings and frustration over the assistant's previous interaction.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Reading comprehension benchmark/test question dataset item rather than an authentic personal disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    notes="Fictional/roleplay dialogue rather than a genuine personal conversation.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|relational|advice_decision",
    notes="Direct first-person disclosure of a profound personal and family disclosure seeking crisis advice.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="relational|advice_decision",
    notes="Direct first-person personal disclosure detailing a moral/relational dilemma around pet custody after a breakup.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Setup prompt for an emotion-guessing exercise/game rather than an organic personal disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="relational|advice_decision",
    notes="Direct first-person advice-seeking query regarding post-breakup communication.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|relational|advice_decision",
    notes="Direct first-person query seeking relational advice to support a depressed close friend.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Jailbreak roleplay setup prompt attempting to bypass restrictions rather than a genuine personal disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="personal_lifestyle|advice_decision",
    notes="Direct first-person advice-seeking query regarding professional/academic correspondence about a letter of recommendation.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|personal_lifestyle|advice_decision",
    notes="Direct first-person personal narrative of severe social anxiety while traveling solo.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="relational|advice_decision",
    notes="Direct first-person user query seeking advice to prevent family arguments during the holidays.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="relational|advice_decision",
    notes="Direct first-person advice request regarding marital relationship dynamics and personal identity.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Scriptwriting/roleplay continuation prompt rather than an authentic direct user disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="relational|personal_lifestyle|advice_decision",
    notes="Direct first-person advice-seeking query regarding military chain of command and leave approval.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional",
    notes="Direct first-person emotional disclosure expressing anxiety.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="personal_lifestyle|advice_decision",
    notes="Direct first-person advice request regarding personal development and self-actualization.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional|personal_lifestyle|advice_decision",
    notes="Direct first-person medical/mental health disclosure and query regarding modafinil and antidepressant effects.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="relational|advice_decision",
    notes="Direct first-person advice request detailing personal relationship struggles and ex-partner oversharing.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Roleplay simulation/character response prompt rather than an authentic personal disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Formatting/summarization task on a transcript rather than an authentic first-person user disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="emotional",
    notes="Direct first-person emotional disclosure expressing sadness and seeking relief.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Academic grading prompt assessing student creative writing against a rubric rather than an authentic personal disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Playful roleplay prompt involving a nonsensical premise rather than a real-world personal disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="relational|advice_decision",
    notes="Direct first-person relationship query seeking advice on managing partner communication dynamics.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="relational|advice_decision",
    notes="Direct first-person query regarding intimate partner relationship dynamics.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="News article summarization task rather than an authentic first-person user disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="relational|emotional|advice_decision",
    notes="Direct first-person query asking for advice and message templates on handling a friend who ghosted.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="relational|personal_lifestyle|advice_decision",
    notes="Direct first-person parenting disclosure and request for a daily schedule for young toddlers.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
label_validation_example(
    current_row,
    relevant=0,
    category="N/A",
    notes="Roleplay menu selection prompt using arbitrary customer trait inputs rather than authentic personal disclosure.",
)

In [ ]:
current_row = show_next_unreviewed()

In [ ]:
validation_df = pd.read_csv(
    validation_path,
    dtype={
        "manual_relevant": "string",
        "manual_category": "string",
        "manual_notes": "string",
    },
)

for column in [
    "manual_relevant",
    "manual_category",
    "manual_notes",
]:
    validation_df[column] = (
        validation_df[column]
        .fillna("")
        .astype("string")
        .str.strip()
    )

validation_df["manual_relevant"] = (
    validation_df["manual_relevant"]
    .replace({
        "1.0": "1",
        "0.0": "0",
    })
)

print("Total validation rows:", len(validation_df))
print("\nManual relevance decisions:")
print(validation_df["manual_relevant"].value_counts(dropna=False).sort_index())

print(
    "\nUnreviewed rows:",
    (~validation_df["manual_relevant"].isin(["0", "1"])).sum()
)

In [ ]:
validated_relevant_df = validation_df[
    validation_df["manual_relevant"] == "1"
].copy()

print("Validated relevant conversations:", len(validated_relevant_df))
print(
    "Validated unique source conversations:",
    validated_relevant_df["source_index"].nunique(),
)

In [ ]:
label_validation_example(
    current_row,
    relevant=1,
    category="relational|advice_decision",
    notes="Direct first-person query seeking actionable ideas/advice for supporting and showing affection to spouse.",
)

In [ ]:
from shutil import copy2

validation_backup_path = validation_path.with_name(
    "lmsys_relevance_manual_validation_200_COMPLETE_backup.csv"
)

copy2(validation_path, validation_backup_path)

print("Backup created:")
print(validation_backup_path)

In [ ]:
RANDOM_SEED = 42
FINAL_PILOT_SIZE = 100

if len(validated_relevant_df) < FINAL_PILOT_SIZE:
    raise ValueError(
        f"Only {len(validated_relevant_df)} validated relevant examples "
        f"available; {FINAL_PILOT_SIZE} required."
    )

pilot_df = (
    validated_relevant_df
    .sample(
        n=FINAL_PILOT_SIZE,
        random_state=RANDOM_SEED,
    )
    .sort_values(
        ["source_index", "pair_index"]
    )
    .reset_index(drop=True)
)

print("Final pilot rows:", len(pilot_df))
print(
    "Unique source conversations:",
    pilot_df["source_index"].nunique(),
)

In [ ]:
assert len(pilot_df) == 100
assert pilot_df["source_index"].nunique() == 100
assert pilot_df["manual_relevant"].eq("1").all()

assert pilot_df["user_text"].notna().all()
assert pilot_df["assistant_text"].notna().all()

print("Final pilot integrity checks passed.")

In [ ]:
pilot_path = (
    PROJECT_ROOT
    / "data"
    / "samples"
    / "lmsys_relevance_pilot_100.csv"
)

pilot_df.to_csv(
    pilot_path,
    index=False,
)

print("Final pilot saved to:")
print(pilot_path)

In [ ]:
validated_relevant_path = (
    PROJECT_ROOT
    / "data"
    / "samples"
    / "lmsys_relevance_validated_relevant_109.csv"
)

validated_relevant_df.to_csv(
    validated_relevant_path,
    index=False,
)

print("Full validated-relevant set saved to:")
print(validated_relevant_path)

In [ ]:
print("=== Filtering and manual validation summary ===")

print("Validation pool:", len(validation_df))
print(
    "Manually relevant:",
    (validation_df["manual_relevant"] == "1").sum(),
)
print(
    "Manually not relevant:",
    (validation_df["manual_relevant"] == "0").sum(),
)

relevance_rate = (
    validation_df["manual_relevant"] == "1"
).mean()

print(
    "Candidate-pool relevance rate:",
    f"{relevance_rate:.1%}",
)

print("Validated relevant pool:", len(validated_relevant_df))
print("Final pilot:", len(pilot_df))
print("Unique pilot sources:", pilot_df["source_index"].nunique())
print("Random seed:", RANDOM_SEED)

In [ ]:
from IPython.display import display, Markdown


def show_validation_example(row_number: int) -> None:
    """
    Display one candidate for manual topical relevance review.
    """

    row = validation_df.iloc[row_number]

    display(
        Markdown(
            f"""
### Candidate {row_number + 1} of {len(validation_df)}

**Source index:** `{row["source_index"]}`  
**Pair index:** `{row["pair_index"]}`  
**Automated category:** `{row["relevance_categories"]}`  
**Redacted:** `{row["redacted"]}`

#### User
{row["user_text"]}

#### Assistant
{row["assistant_text"]}

---

**Manual decision:**  
`1` = genuinely relevant  
`0` = not relevant
"""
        )
    )

In [ ]:
show_validation_example(0)

def show_next_unreviewed():
    """
    Display the next candidate that has not yet been manually reviewed.
    """

    unreviewed = validation_df[
        ~validation_df["manual_relevant"].isin(["0", "1"])
    ]

    if unreviewed.empty:
        print("All candidates have been reviewed.")
        return None

    row_number = unreviewed.index[0]

    show_validation_example(row_number)

    print(f"\nRow number to label: {row_number}")

    return row_number

In [ ]:
current_row = show_next_unreviewed() 

In [ ]:
def label_validation_example(
    row_number: int,
    relevant: int,
    category: str = "",
    notes: str = "",
) -> None:
    """
    Record a manual topical-relevance decision and save immediately.

    relevant:
        1 = genuinely relevant
        0 = not relevant
    """

    if relevant not in (0, 1):
        raise ValueError("relevant must be either 0 or 1")

    validation_df.at[row_number, "manual_relevant"] = str(relevant)
    validation_df.at[row_number, "manual_category"] = category
    validation_df.at[row_number, "manual_notes"] = notes

    validation_df.to_csv(
        validation_path,
        index=False,
    )

    print(
        f"Saved row {row_number + 1}: "
        f"manual_relevant={relevant}"
    )

In [ ]:
label_validation_example(
    0,
    relevant=1,
    category="relational|advice_decision",
    notes="Genuine personal relationship/advice request",
)

In [ ]:
print("Candidate pairs:", len(candidate_df))

print(
    "Unique source conversations:",
    candidate_df["source_index"].nunique(),
)

print("\nPairs per source conversation:")
print(
    candidate_df["source_index"]
    .value_counts()
    .value_counts()
    .sort_index()
)

print("\nRedacted flag counts:")
print(
    candidate_df["redacted"]
    .value_counts(dropna=False)
)

In [ ]:
pd.set_option("display.max_colwidth", 300)

inspection_df = candidate_df.sample(
    n=min(20, len(candidate_df)),
    random_state=RANDOM_SEED,
)

inspection_df[
    [
        "source_index",
        "pair_index",
        "user_text",
        "assistant_text",
        "relevance_categories",
    ]
]

In [ ]:
print("Diagnostics:")
print(diagnostics)

print("\nCategory combinations:")
print(
    candidate_df["relevance_categories"]
    .value_counts()
) 